# Preprocessing

### Kontext a logika nazvov
Dataset ma zmatocne nazvy stlpcov, vztahy medzi stlpcami jednotlivych datasetov su:

- `books.id` (1–10000, interne (pri vytvarani datasetu) record ID) ↔ `ratings.book_id`
- `books.book_id` (goodreads ID, ako sme ukazali v EDA) ↔ `book_tags.goodreads_book_id`
- `tags.tag_id` ↔ `book_tags.tag_id`

### Nacitanie dat

In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import os

In [2]:
books_df = pd.read_csv('./goodbooks_ds/books.csv')
ratings_df = pd.read_csv('./goodbooks_ds/ratings.csv')
book_tags_df = pd.read_csv('./goodbooks_ds/book_tags.csv')
tags_df = pd.read_csv('./goodbooks_ds/tags.csv')

In [3]:
print(f"books_df shape: {books_df.shape}")
print(f"ratings_df shape: {ratings_df.shape}")
print(f"book_tags_df shape: {book_tags_df.shape}")
print(f"tags_df shape: {tags_df.shape}")

books_df shape: (10000, 23)
ratings_df shape: (981756, 3)
book_tags_df shape: (999912, 3)
tags_df shape: (34252, 2)


## Cistenie books datasetu

- `original_title` ma 585 nullov, ktore nahradime nazvami zo stlpca `title` (nema null hodnoty)
- `language_code` ma 1084 nullov, nahradime `'unknown'`
- `original_publication_year` ma 21 nullov, tieto hodnoty nemame ako doplnit, nechame ich tak
- `isbn`, `isbn13` maju nulls, pre content filtering a odporucania nie su potrebne, ale podobne ako pri language_code ich mozme nahradit 'unknown'
- Premenovanie stlpcov `id` na `record_id`, `book_id` na `goodreads_book_id`

In [4]:
print("Počet NULL hodnôt pred čistením: ")
print(f"original_title: {books_df['original_title'].isnull().sum()}")
print(f"language_code: {books_df['language_code'].isnull().sum()}")

#publication year nakoniec nebudeme upravovat
#print(f"original_publication_year: {books_df['original_publication_year'].isnull().sum()}")
print(f"isbn: {books_df['isbn'].isnull().sum()}")
print(f"isbn13: {books_df['isbn13'].isnull().sum()}")

Počet NULL hodnôt pred čistením: 
original_title: 585
language_code: 1084
isbn: 700
isbn13: 585


In [5]:
books_df['original_title'] = books_df['original_title'].fillna(books_df['title'])
books_df['language_code'] = books_df['language_code'].fillna('unknown')
books_df['isbn'] = books_df['isbn'].fillna('unknown')
books_df['isbn13'] = books_df['isbn13'].fillna('unknown')

Anglicke knihy zjednotime pod *eng* (iba pre zjednodusenie): 

In [6]:
english_variants = ['eng', 'en-US', 'en-GB', 'en-CA', 'en-AU', 'en-IN', 'en']

books_df['language_code'] = books_df['language_code'].apply(
    lambda x: 'eng' if x in english_variants else x
)

print(books_df['language_code'].value_counts())

language_code
eng        8730
unknown    1084
ara          64
fre          25
ind          21
spa          20
ger          13
jpn           7
per           7
pol           6
por           6
nor           3
dan           3
fil           2
ita           2
nl            1
rum           1
mul           1
tur           1
swe           1
vie           1
rus           1
Name: count, dtype: int64


In [7]:
books_df = books_df.rename(columns={'id': 'record_id', 'book_id': 'goodreads_book_id'})

In [8]:
print("Počet NULL hodnôt po čistení:")
print(f"original_title: {books_df['original_title'].isnull().sum()}")
print(f"language_code: {books_df['language_code'].isnull().sum()}")
print(f"isbn: {books_df['isbn'].isnull().sum()}")
print(f"isbn13: {books_df['isbn13'].isnull().sum()}")

print("\nNové stĺpce: record_id, goodreads_book_id")
print(f"books_df shape: {books_df.shape}")

Počet NULL hodnôt po čistení:
original_title: 0
language_code: 0
isbn: 0
isbn13: 0

Nové stĺpce: record_id, goodreads_book_id
books_df shape: (10000, 23)


## Cistenie book_tags_df

- Odstranit riadky kde `count < 0` (existuje 6 riadkov s count = -1)
- Odstranit duplicitne riadky (6 plnych duplicit, 8 kombinovanych s goodreads_book_id + tag_id), ponechame prvy vyskyt
Rozhodli sme sa ponechat prvy vyskyt, pretoze vsetky az na 1 duplicitu maju rovnaky pocet count, pri tom jednom pripade je count iba o 1 vacsi.

In [9]:
print(f"book_tags_df initial shape: {book_tags_df.shape}")
print(f"Riadky s count < 0: {(book_tags_df['count'] < 0).sum()}")
print(f"Duplicitné riadky: {book_tags_df.duplicated().sum()}")
print(f"Duplicita podľa (goodreads_book_id, tag_id): {book_tags_df.duplicated(subset=['goodreads_book_id', 'tag_id']).sum()}")

book_tags_df initial shape: (999912, 3)
Riadky s count < 0: 6
Duplicitné riadky: 6
Duplicita podľa (goodreads_book_id, tag_id): 8


In [10]:
book_tags_df = book_tags_df[book_tags_df['count'] >= 0]
print(f"Po odstránení zaporných count: {book_tags_df.shape}")

Po odstránení zaporných count: (999906, 3)


In [11]:
book_tags_df = book_tags_df.drop_duplicates(keep='first')
print(f"Po odstránení duplikátov: {book_tags_df.shape}")

Po odstránení duplikátov: (999900, 3)


## Cistenie tags_df

- Odstranit šumove tagy: nechame iba riadky, kde `tag_name` zacina pismenom, nie je "filler" tag (currently-reading, to-read, favorite..)

In [12]:
print(f"tags_df zaciatocny shape: {tags_df.shape}")
print("Ukážka tagov pred čistením:")
print(tags_df.head(10))

tags_df zaciatocny shape: (34252, 2)
Ukážka tagov pred čistením:
   tag_id tag_name
0       0        -
1       1     --1-
2       2    --10-
3       3    --12-
4       4   --122-
5       5   --166-
6       6    --17-
7       7    --19-
8       8     --2-
9       9   --258-


In [13]:
tags_df = tags_df[tags_df['tag_name'].str.match(r'^[a-zA-Z]', na=False)]

Z tags odstranime aj stlpce ako to-read, currently-reading, favorites ap, ktore sme nasli v EDA. Tieto stlpce nemaju ziadnu vypovednu hodnotu o knihe, teda o jej content-e. Pri preprocessingu sme ich nasli este ovela viac pri prezerani vytovrene mergnuteho datasetu ako *to-buy, own, mine..*

In [14]:
#iba konkretne tags
#tags_df = tags_df[~tags_df['tag_name'].isin(["to-read", "favorites", "currently-reading","favorite","reading"])]

#vsetky tags s vyskytom key words
tags_df = tags_df[~tags_df['tag_name'].str.contains("read|to-buy|to-sell|fav|owned|i-own|mine|my-books|own-it|star-rating", case=False)]

In [15]:
print(f"tags_df final shape: {tags_df.shape}")

tags_df final shape: (29500, 2)


In [16]:
print(f"Odstránené tagy: {34252 - len(tags_df)}")
print("Ukážka očistených tagov:")
print(tags_df.head(10))

Odstránené tagy: 4752
Ukážka očistených tagov:
      tag_id                  tag_name
1291    1291                         a
1293    1293               a-a-fiction
1294    1294                 a-a-milne
1295    1295               a-amsterdam
1296    1296     a-bad-case-of-stripes
1297    1297       a-baker-s-education
1298    1298          a-beautiful-dark
1303    1303         a-child-called-it
1304    1304                a-christie
1305    1305  a-corner-of-the-universe


Mozeme si vsimnut tag 'a', nie sme si isty co to znamena, mozno to ma nejaky vyznam, o ktorom nevieme, pri niektorych knihach sa vyskytoval celkom velakrat. Mozno ako tag ya (young adult) znamena adult, preto sme sa ho rozhodli nechat.

In [17]:
book_tags_df[book_tags_df['tag_id'] == tags_df[tags_df['tag_name'] == 'a']['tag_id'].values[0]]['count'].max()

np.int64(19)

## Cistenie ratings_df

- Odstranit duplicitne dvojice (`user_id`, `book_id`), ponechat riadok s vyssim ratingom

In [18]:
print(f"ratings_df initial shape: {ratings_df.shape}")
print(f"Duplikáty podľa (user_id, book_id): {ratings_df.duplicated(subset=['user_id', 'book_id']).sum()}")

ratings_df initial shape: (981756, 3)
Duplikáty podľa (user_id, book_id): 2278


In [19]:
ratings_df = ratings_df.sort_values('rating', ascending=False).drop_duplicates(subset=['user_id', 'book_id'], keep='first')
ratings_df = ratings_df.sort_index().reset_index(drop=True)
print(f"ratings_df final shape: {ratings_df.shape}")

ratings_df final shape: (979478, 3)


Rozdhodli sme sa nechat vyssi rating pre danu knihu od daneho pouzivatela. Kedze nevieme, ako tato duplicita vznikla, a nas dataset nema timestamps, takze nevieme, ktore hodnotenie prislo ako prve, rozhodli sme sa nechat vyssi rating, pretoze sa zameriavame na to, co sa pouzivatelovi pacilo. Zaroven, ~2000 chybnych ratingov nie je moc signifikantny pri ~1 000 000 celkovych ratingov.

## Vytvorenie tag profilu pre kazdu knihu

- Mergneme `book_tags` s `tags` cez `tag_id`
- Pre kazdu knihu vybrať top 50 tagov podla `count`
- Spojime `tag_name` do jedneho retazca oddeleneho medzerou (inspiracia z jupyter notebooku z cvicenia 2)

Povodne sme brali iba top 20 tagov, ale zistili sme, ze v top tagoch sa nachadzaju "filler" tagy, ktore nemaju nic spolocne s obsahom knihy. Tento problem vznika, pretoze goodreads dovoluje pouzivatelom pridat vlastne tagy, nie su teda vzdy jednotne. Kvoli tomuto sa niekedy v top 20 vyskytovali tagy ako "favourite". Aj ked odstranime vsetky tagy so sub-retazcom "fav" alebo "read", nezarucuje to, ze sme odstranili vsetky taketo filler tagy. Preto sme sa rozhodli pre vacsi pocet tagov, aby mala kniha co najviac relevantnych tags aj keby sme neodstranili niektore filler tagy. 

Zaroven, ako sme uz ukazali v EDA, niektore tagy, aj ked s mensim poctom, maju velku vypovednu hodnotu, napr. tag ako *sci-fi* bolo v tagoch s mensim poctom, ale hovori vela o zanre knihy. Nechceli sme zaratat vsetky tagy, kvoli uz spominanej irrelevancii, ale prilis malo tagov (ako napr 20, co sme skusali) sposobovalo, ze knihy nemali moc velku variabilitu tagov. 

In [20]:
merged_tags = book_tags_df.merge(tags_df, on='tag_id')
print(f"merged_tags shape: {merged_tags.shape}")

merged_tags shape: (736745, 4)


In [21]:
top_tags = (merged_tags
    .sort_values(['goodreads_book_id', 'count'], ascending=[True, False])
    .groupby('goodreads_book_id')
    .head(50))
print(f"top_tags shape: {top_tags.shape}")

top_tags shape: (499429, 4)


In [22]:
tag_profiles = (merged_tags
    .groupby('goodreads_book_id')['tag_name']
    .apply(lambda x: ' '.join(x))
    .reset_index()
    .rename(columns={'tag_name': 'tags_string'}))

print(f"tag_profiles shape: {tag_profiles.shape}")
print("\nUkážka tag_profiles:")
print(tag_profiles.head())

tag_profiles shape: (10000, 2)

Ukážka tag_profiles:
   goodreads_book_id                                        tags_string
0                  1  fantasy young-adult fiction harry-potter ya se...
1                  2  fantasy children children-s default fiction yo...
2                  3  fantasy young-adult fiction harry-potter ya se...
3                  5  fantasy young-adult fiction harry-potter ya se...
4                  6  fantasy young-adult fiction harry-potter ya se...


Jedny z prvych knih s goodreads_book_id su harry potter knihy, preto maju rovnake tags.

## Vytvorenie hlavnej feature matice books_content_df

Vytvorenie finálneho dataframu s potrebnymi stlpcami pre content-based filtering:

In [23]:
books_content_df = books_df[['record_id', 'goodreads_book_id', 'title','original_title', 'authors', 'average_rating',
                             'original_publication_year', 'language_code']].copy()

books_content_df = books_content_df.merge(tag_profiles, on='goodreads_book_id', how='left')

In [24]:
nan_count = books_content_df['tags_string'].isnull().sum()
print(f"Knihy s tags_string == NaN: {nan_count}")
print(f"books_content_df shape: {books_content_df.shape}")
print("\nbooks_content_df.head():")
print(books_content_df.head())
print("\nbooks_content_df.info():")
books_content_df.info()

Knihy s tags_string == NaN: 0
books_content_df shape: (10000, 9)

books_content_df.head():
   record_id  goodreads_book_id  \
0          1            2767052   
1          2                  3   
2          3              41865   
3          4               2657   
4          5               4671   

                                               title  \
0            The Hunger Games (The Hunger Games, #1)   
1  Harry Potter and the Sorcerer's Stone (Harry P...   
2                            Twilight (Twilight, #1)   
3                              To Kill a Mockingbird   
4                                   The Great Gatsby   

                             original_title                      authors  \
0                          The Hunger Games              Suzanne Collins   
1  Harry Potter and the Philosopher's Stone  J.K. Rowling, Mary GrandPré   
2                                  Twilight              Stephenie Meyer   
3                     To Kill a Mockingbird              

## Ulozenie vystupov
Nove datasety ulozime ako csv subory, aby sme s nimi v *projekt1_recsys.ipynb* mohli dalej pracovat.

In [25]:
os.makedirs('./preprocessed', exist_ok=True)
books_content_df.to_csv('./preprocessed/books_content.csv', index=False)
ratings_df.to_csv('./preprocessed/ratings_clean.csv', index=False)

print("Uložené súbory:")
print("  - ./preprocessed/books_content.csv")
print("  - ./preprocessed/ratings_clean.csv")
print(f"\nZhrnutie:")
print(f"  books_content_df: {books_content_df.shape[0]} kníh, {books_content_df.shape[1]} stĺpcov")
print(f"  ratings_clean_df: {ratings_df.shape[0]} hodnotení")

Uložené súbory:
  - ./preprocessed/books_content.csv
  - ./preprocessed/ratings_clean.csv

Zhrnutie:
  books_content_df: 10000 kníh, 9 stĺpcov
  ratings_clean_df: 979478 hodnotení


## Zhrnutie celého preprocessing procesu

| Krok | Dataset | Pociatocny shape | Koncovy shape | Zmeny |
|------|---------|-----------------|---------------|-------|
| 2 | books_df | (10000, 23) | (10000, 23) | Vyplnenie null, premenovanie stlpcov |
| 3 | book_tags_df | (999912, 3) | (999900, 3) | Odstranenie duplikatov, tagov s count=-1|
| 4 | tags_df | (34252, 2) | (29500, 2) | Odstranenie šumu - *favorite, to-read, -185-..* |
| 5 | ratings_df | (981756, 3) | (979478, 3) | Odstranenie duplikatov |
| 6 | tag_profiles | - | (10000, 2) | Vytvorenie tag profilov (top 50) |
| 7 | books_content_df | - | (10000, 9) | Finalny dataset (merge) |
